# 从静态神经场走向动态世界

NeRF 的第一步是学一个函数：给定空间坐标，交出密度和颜色。我们先拟合一颗彩色小球，再说明时间与动作怎样进入接口。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.spatial import (
    TinyDynamicField, TinyNeuralField, make_colored_sphere_samples,
    make_moving_sphere_samples,
)
torch.manual_seed(0)


## 1. 坐标不是图片编号

训练样本是 `(x,y,z) → density,color`。同一个连续函数可以在没有见过的坐标上查询。

In [ ]:
coordinates, density, color = make_colored_sphere_samples(640, seed=0)
print('coordinates/density/color:', tuple(coordinates.shape), tuple(density.shape), tuple(color.shape))
assert coordinates.shape[1] == 3


## 2. 拟合最小神经场

密度说明射线在哪里遇到物体，颜色说明该处呈现什么。真正 NeRF 还会沿相机射线采样并做体渲染。

In [ ]:
field = TinyNeuralField()
opt = torch.optim.Adam(field.parameters(), lr=5e-3)
losses = []
for _ in range(80):
    opt.zero_grad(); predicted_density, predicted_color = field(coordinates); loss = torch.nn.functional.mse_loss(predicted_density, density) + torch.nn.functional.mse_loss(predicted_color, color); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('field loss:', round(losses[0], 3), '→', round(losses[-1], 3))
assert losses[-1] < losses[0]


## 3. 加入时间与动作

静态函数只有 `(x,y,z)`。现在生成一颗会随时间和动作移动的小球，并学习 `(x,y,z,t,action) → density,color`。这仍是坐标查询 toy，不是完整 4D 重建。

In [ ]:
coordinates_4d, times, actions, density_4d, color_4d = (
    make_moving_sphere_samples(1024, seed=1)
)
dynamic_field = TinyDynamicField()
opt = torch.optim.Adam(dynamic_field.parameters(), lr=5e-3)
dynamic_losses = []
for _ in range(100):
    predicted_density, predicted_color = dynamic_field(
        coordinates_4d, times, actions
    )
    loss = (
        torch.nn.functional.binary_cross_entropy(
            predicted_density, density_4d
        )
        + torch.nn.functional.mse_loss(predicted_color, color_4d)
    )
    opt.zero_grad(); loss.backward(); opt.step()
    dynamic_losses.append(float(loss.detach()))
print('dynamic loss:', round(dynamic_losses[0], 3), '→',
      round(dynamic_losses[-1], 3))
assert dynamic_losses[-1] < dynamic_losses[0]


## 4. 固定坐标，只换时间或动作

动作条件模型至少应在条件变化时给出不同的查询结果。差异本身还不证明运动正确，但结果完全相同一定有问题。

In [ ]:
query = torch.tensor([[0.35, 0.0, 0.0]]).expand(5, -1)
with torch.no_grad():
    early_density, _ = dynamic_field(
        query, torch.zeros(5), torch.arange(5)
    )
    late_density, _ = dynamic_field(
        query, torch.ones(5), torch.arange(5)
    )
print('t=0 density:', [round(float(x), 3) for x in early_density])
print('t=1 density:', [round(float(x), 3) for x in late_density])
difference = (early_density - late_density).abs().mean()
assert difference > 0


## 5. NeRF、3DGS 与 Mesh 的选择

NeRF 用连续网络查询，适合高质量新视角；3DGS 用显式高斯快速渲染；Mesh 用三角形提供明确表面，便于编辑与碰撞。它们是空间表示，不自动包含动作动态。

In [ ]:
representations = {'NeRF': '连续场，查询慢但平滑', '3DGS': '显式高斯，渲染快', 'Mesh': '明确表面，方便碰撞'}
for name, purpose in representations.items(): print(f'{name:5s}: {purpose}')


## 小结

这份 Notebook 交出静态场与动作条件动态场两个最小接口。它证明时间和动作可以进入查询，也展示反事实差异；它仍没有相机射线、体渲染、多视角重建和真实物理。PA1-E2a 再完成这些部分。